[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/06_Datasets_Benchmarks/01_multimodal_datasets/01_multimodal_datasets.ipynb)

# 01. Multimodal Datasets & Data Curation

**This notebook covers:**
- Load and explore a sample dataset from HuggingFace
- Analyze data distributions
- Data preprocessing pipelines
- Compare dataset sizes and characteristics

**Runtime:** ~10–15 minutes on CPU

---

> **Theory & derivations:** See [README.md](./README.md) for full step-by-step math.


In [ ]:
# ============================================================
#  Google Colab Setup — Run this cell FIRST
# ============================================================
import os, sys

try:
    import google.colab
    IN_COLAB = True
    print("Google Colab detected — setting up environment...")
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        print("Cloning repository...")
        !git clone --depth 1 {REPO_URL} {REPO_DIR}
    else:
        print("Repository already cloned")

    print("Installing dependencies...")
    !pip install -q -r {REPO_DIR}/requirements.txt

    MODULE_DIR = f"{REPO_DIR}/06_Datasets_Benchmarks/01_multimodal_datasets"
    os.chdir(MODULE_DIR)
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    print(f"Colab setup complete — {os.getcwd()}")

    import torch
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("Device: CPU (all notebooks work fine on CPU)")
else:
    os.makedirs("../../assets", exist_ok=True)
    print("Running locally — all set!")

In [ ]:
import sys
sys.path.append('../..')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

try:
    from utils.visualization import set_style
    from utils.helpers import count_parameters, get_device
    set_style()
except ImportError:
    def set_style():
        plt.rcParams.update({'figure.figsize': (10, 6), 'figure.dpi': 100})
    def count_parameters(model):
        total = sum(p.numel() for p in model.parameters())
        print(f"Total parameters: {total:,}")
        return total
    def get_device():
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    set_style()

torch.manual_seed(42)
np.random.seed(42)
device = get_device() if callable(get_device) else torch.device('cpu')
print(f"PyTorch {torch.__version__} | Device: {device}")

## 1. Dataset Landscape Overview


In [ ]:
PRETRAIN_DATASETS = {
    "LAION-5B": {"pairs": "5.85B", "modality": "image-text", "notes": "web crawl, CLIP-filtered"},
    "CC3M": {"pairs": "3.3M", "modality": "image-text", "notes": "Conceptual Captions"},
    "CC12M": {"pairs": "12.4M", "modality": "image-text", "notes": "noisier alt-text"},
    "DataComp": {"pairs": "variable", "modality": "image-text", "notes": "competition-scale filtering"},
    "WebLI": {"pairs": "10B+", "modality": "image-text", "notes": "Google scale pretraining"},
}

INSTRUCT_DATASETS = {
    "LLaVA-Instruct": {"size": "150K", "type": "instruction tuning"},
    "ShareGPT4V": {"size": "100K+", "type": "GPT-4V conversations"},
    "SVIT": {"size": "4.8M", "type": "synthetic VLM instructions"},
}

EVAL_BENCHMARKS = {
    "MME": "Perception + cognition score",
    "MMMU": "Multi-discipline multimodal understanding",
    "MM-Bench": "Structured VQA capabilities",
    "SEED-Bench": "19 dimensions of LVLM evaluation",
}

for name, info in PRETRAIN_DATASETS.items():
    print(f"{name:12} {info}")

## 2. Load Sample Dataset (Synthetic + Optional HF)


In [ ]:
# Synthetic caption dataset (always works offline)
synthetic = [
    {"image_id": i, "caption": cap, "source": "synthetic", "tokens": len(cap.split())}
    for i, cap in enumerate([
        "a cat on a mat", "dog playing fetch", "sunset over mountains",
        "city street at night", "person riding a bicycle", "bowl of fresh fruit",
        "snow covered trees", "children in a playground", "coffee on a wooden table",
        "airplane in the sky",
    ] * 3)
]
print(f"Synthetic samples: {len(synthetic)}")
print("Example:", synthetic[0])

# Optional HuggingFace load
try:
    from datasets import load_dataset
    ds = load_dataset("nlphuji/flickr30k", split="test[:20]", trust_remote_code=True)
    print(f"\nFlickr30k test subset: {len(ds)} rows, columns={ds.column_names}")
    hf_ok = True
except Exception as e:
    print("\nHF load skipped (offline or missing):", e)
    ds = None
    hf_ok = False

## 3. Data Distribution Analysis


In [ ]:
lengths = [s["tokens"] for s in synthetic]
sources = Counter(s["source"] for s in synthetic)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(lengths, bins=8, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Caption length (tokens)'); axes[0].set_title('Caption Length Distribution')
axes[1].bar(sources.keys(), sources.values(), color='coral')
axes[1].set_title('Samples by Source')
plt.tight_layout(); plt.show()

if hf_ok and ds is not None:
    hf_lens = [len(row["caption"].split()) if "caption" in row else len(row["sentences"][0].split()) for row in ds]
    print(f"HF caption length mean={np.mean(hf_lens):.1f}, std={np.std(hf_lens):.1f}")

## 4. Preprocessing Pipeline


In [ ]:
def preprocess_batch(examples, max_length=16):
    out = []
    for ex in examples:
        cap = ex.get("caption", "")
        cap = cap.lower().strip()
        cap = " ".join(cap.split())  # normalize whitespace
        tokens = cap.split()[:max_length]
        out.append({"caption_clean": " ".join(tokens), "length": len(tokens), "image_id": ex["image_id"]})
    return out

clean = preprocess_batch(synthetic)
print("Preprocessed sample:", clean[0])

# Deduplication demo
unique_caps = {c["caption_clean"] for c in clean}
print(f"Before dedup: {len(clean)} -> unique captions: {len(unique_caps)}")

## 5. Quality Scoring & Filtering


In [ ]:
def quality_score(caption):
    tokens = caption.split()
    if len(tokens) < 3:
        return 0.2
    score = 0.5
    if len(tokens) >= 5:
        score += 0.2
    if any(w in caption for w in ["a", "the", "on", "in"]):
        score += 0.1
    return min(score, 1.0)

scored = [(c["caption_clean"], quality_score(c["caption_clean"])) for c in clean]
scored.sort(key=lambda x: -x[1])
print("Top quality captions:")
for cap, s in scored[:5]:
    print(f"  [{s:.2f}] {cap}")

threshold = 0.6
filtered = [c for c in clean if quality_score(c["caption_clean"]) >= threshold]
print(f"\nKept {len(filtered)}/{len(clean)} after quality filter (>={threshold})")

## 6. Dataset Size Comparison


In [ ]:
comparison = [
    ("LAION-5B", 5.85e9, "pretrain"),
    ("WebLI", 10e9, "pretrain"),
    ("CC12M", 12.4e6, "pretrain"),
    ("LLaVA-Instruct", 150e3, "instruct"),
    ("ShareGPT4V", 100e3, "instruct"),
    ("Flickr30k", 31.8e3, "eval"),
]

names, sizes, kinds = zip(*comparison)
colors = ['#4C72B0' if k == 'pretrain' else '#55A868' if k == 'instruct' else '#C44E52' for k in kinds]
plt.figure(figsize=(10, 5))
plt.barh(names, sizes, color=colors)
plt.xscale('log'); plt.xlabel('Number of image-text pairs (log scale)')
plt.title('Multimodal Dataset Scale Comparison'); plt.tight_layout(); plt.show()

## Summary

Explored pretraining/instruction/eval datasets, built a preprocessing and quality filtering pipeline, and compared dataset scales.

See [README.md](./README.md) for full dataset catalog and curation math.
